In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Imbalanced data
from imblearn.over_sampling import SMOTE

# Classical ML models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Model persistence
import joblib

# Misc
import warnings
warnings.filterwarnings('ignore')

print(f'pandas     {pd.__version__}')
print(f'numpy      {np.__version__}')
print(f'sklearn    {__import__("sklearn").__version__}')
print(f'tensorflow {tf.__version__}')
print(f'seaborn    {sns.__version__}')

In [ ]:
# ---------------------------------------------------------------------------
# Zelle 1: Daten laden / Load raw data
# ---------------------------------------------------------------------------

TRAIN_PATH = "Data/fraudTrain.csv"
TEST_PATH  = "Data/fraudTest.csv"

df_train = pd.read_csv(TRAIN_PATH, index_col=0)
df_test  = pd.read_csv(TEST_PATH,  index_col=0)

print(f"Train shape : {df_train.shape}")
print(f"Test  shape : {df_test.shape}")
print(f"\nFraud rate (train): {df_train['is_fraud'].mean():.4%}")
print(f"Fraud rate (test) : {df_test['is_fraud'].mean():.4%}")
print(f"\nColumns:\n{df_train.columns.tolist()}")
print(f"\nNull counts (train):\n{df_train.isnull().sum()[df_train.isnull().sum() > 0]}")

In [ ]:
# ---------------------------------------------------------------------------
# Zelle 2: Feature-Engineering
# ---------------------------------------------------------------------------

def haversine_km(lat1, lon1, lat2, lon2):
    """Vektorisierte Haversine-Distanz in Kilometern."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def engineer_features(df):
    """Erstellt neue Features. Gibt eine Kopie zurueck (unveraenderlich)."""
    df = df.copy()

    # Zeitstempel-Features
    ts = pd.to_datetime(df["trans_date_trans_time"])
    df["hour"]        = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek   # 0=Montag ... 6=Sonntag
    df["month"]       = ts.dt.month

    # Altersberechnung (festes Referenzdatum = letzter Tag im Datensatz)
    REFERENCE_DATE = pd.Timestamp("2020-06-21")
    df["age"] = ((REFERENCE_DATE - pd.to_datetime(df["dob"])).dt.days / 365.25).round(1)

    # Geografische Distanz Karteninhaber <-> Haendler
    df["distance_km"] = haversine_km(
        df["lat"], df["long"], df["merch_lat"], df["merch_long"]
    )

    # Nicht benoetigte Spalten entfernen
    cols_to_drop = [
        "trans_date_trans_time", "cc_num", "first", "last",
        "street", "city", "state", "zip",
        "lat", "long", "merch_lat", "merch_long",
        "trans_num", "unix_time", "dob",
    ]
    return df.drop(columns=cols_to_drop)


df_train_eng = engineer_features(df_train)
df_test_eng  = engineer_features(df_test)

print(f"Train nach Engineering: {df_train_eng.shape}")
print(f"Columns: {df_train_eng.columns.tolist()}")
print(df_train_eng.dtypes)

In [ ]:
# ---------------------------------------------------------------------------
# Zelle 3: Kodierung, Skalierung und numpy-Export
# ---------------------------------------------------------------------------

CATEGORICAL_COLS = ["merchant", "category", "gender", "job"]
TARGET = "is_fraud"

# --- LabelEncoder: nur auf Train fitten; Test mit -1-Fallback transformieren ---
encoders = {}
df_train_enc = df_train_eng.copy()
df_test_enc  = df_test_eng.copy()

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    le.fit(df_train_enc[col])
    encoders[col] = le

    df_train_enc[col] = le.transform(df_train_enc[col])

    # Unbekannte Kategorien im Test-Set -> -1
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    df_test_enc[col] = df_test_enc[col].map(mapping).fillna(-1).astype(int)

    unseen = (df_test_enc[col] == -1).sum()
    print(f"  {col:12s}: {len(le.classes_)} Train-Klassen | {unseen} unbekannt->-1")

# --- X / y trennen ---
FEATURE_COLS = [c for c in df_train_enc.columns if c != TARGET]

X_train = df_train_enc[FEATURE_COLS]
y_train = df_train_enc[TARGET]
X_test  = df_test_enc[FEATURE_COLS]
y_test  = df_test_enc[TARGET]

print(f"\nX_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

# --- StandardScaler: nur auf Train fitten ---
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train)   # np.ndarray float64
X_test_np  = scaler.transform(X_test)        # np.ndarray float64
y_train_np = y_train.to_numpy(dtype=np.int32)
y_test_np  = y_test.to_numpy(dtype=np.int32)

# --- Abschlussbericht ---
print("\n=== Finale numpy-Arrays ===")
print(f"X_train_np : {X_train_np.shape}  dtype={X_train_np.dtype}")
print(f"X_test_np  : {X_test_np.shape}   dtype={X_test_np.dtype}")
print(f"y_train_np : {y_train_np.shape}  fraud={y_train_np.sum()} ({y_train_np.mean():.4%})")
print(f"y_test_np  : {y_test_np.shape}   fraud={y_test_np.sum()} ({y_test_np.mean():.4%})")

# Ungeskalte DataFrames bleiben fuer EDA erhalten: X_train, X_test, y_train, y_test